In [2]:
from Booleanize_global import *

In [3]:
import pandas as pd
import glob
import os

csv_files = glob.glob("/home/ubuntu/local/fractal_maps_isic/*.csv")
filenames = [os.path.splitext(os.path.basename(f))[0].replace('_fd', '') for f in csv_files]
print("Total fractal maps:", len(csv_files))
print(filenames[:5])

Total fractal maps: 25331
['ISIC_0026391', 'ISIC_0031121', 'ISIC_0070032', 'ISIC_0059135', 'ISIC_0066044']


In [4]:
from sklearn.model_selection import train_test_split

x_train, x_test = train_test_split(filenames, test_size=0.2, random_state=21)

In [5]:
all_train_values = []

for f in x_train:
    
    path = f"/home/ubuntu/local/fractal_maps_isic/{f}_fd.csv"
    
    vals = np.loadtxt(path, delimiter=",").flatten()
    
    all_train_values.extend(vals)

print("Collected training fractal values:", len(all_train_values))

Collected training fractal values: 77327424


In [7]:
discretizer = fit_global_bins(all_train_values, n_bins=5)

print("Global quantile bins fitted.")

Global quantile bins fitted.


/home/ubuntu/local/fractal_env/lib/python3.12/site-packages/sklearn/preprocessing/_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


In [8]:
print("Global Quantile Bin Boundaries:")
bins = discretizer.bin_edges_[0]

for i in range(len(bins)-1):
    print(f"Bin {i}: {bins[i]:.9f} → {bins[i+1]:.9f}")

Global Quantile Bin Boundaries:
Bin 0: 0.837300000 → 1.798614000
Bin 1: 1.798614000 → 1.978320000
Bin 2: 1.978320000 → 2.098200000
Bin 3: 2.098200000 → 2.265690000
Bin 4: 2.265690000 → 3.718350000


In [9]:
booleanized_data = {}

for f in filenames:

    path = f"/home/ubuntu/local/fractal_maps_isic/{f}_fd.csv"
    vals = np.loadtxt(path, delimiter=",").flatten()

    booleanized_data[f] = booleanize_array(vals, discretizer).flatten()

print("Booleanization completed.")

Booleanization completed.


In [11]:
print(len(booleanized_data))

25331


In [12]:
sample_key = list(booleanized_data.keys())[0]
print(booleanized_data[sample_key].shape)

(19080,)


In [13]:
cancerous = ['MEL', 'BCC', 'SCC', 'AK']
non_canc = ['NV', 'BKL', 'DF', 'VASC']

In [14]:
import pandas as pd

df = pd.read_csv("/home/ubuntu/Downloads/ISIC2019/ISIC_2019_Training_GroundTruth.csv")

# create cancer vs non-cancer label
df["label"] = (df[['MEL','BCC','SCC','AK']].sum(axis=1) > 0).astype(int)

# create dictionary mapping image → label
label_dict = dict(zip(df['image'], df['label']))

print(len(label_dict))  
print(list(label_dict.items())[:5])

25331
[('ISIC_0000000', 0), ('ISIC_0000001', 0), ('ISIC_0000002', 1), ('ISIC_0000003', 0), ('ISIC_0000004', 1)]


In [15]:
print(df["label"].value_counts())

label
0    15991
1     9340
Name: count, dtype: int64


In [16]:
print(set(len(booleanized_data[f]) for f in filenames))

{19080}


In [17]:
for f in x_train[:20]:
    print(len(booleanized_data[f]))

19080
19080
19080
19080
19080
19080
19080
19080
19080
19080
19080
19080
19080
19080
19080
19080
19080
19080
19080
19080


In [18]:
# for _, row in df.iterrows():
#     image_id = row['image_id']   # match with your filenames
#     diagnosis = row['dx']
#     if diagnosis in cancerous:
#         label_dict[image_id] = 1
#     else:
#         label_dict[image_id] = 0

X_train = np.array([booleanized_data[f] for f in x_train]) # dtype = np.int32 , this is chatgpt reccomendation, try later
X_test = np.array([booleanized_data[f] for f in x_test])

Y_train = np.array([label_dict[f] for f in x_train])
Y_test = np.array([label_dict[f] for f in x_test])

np.save('X_train.npy', X_train, allow_pickle=True)
np.save('X_test.npy', X_test, allow_pickle=True)
np.save('Y_train.npy', Y_train)
np.save('Y_test.npy', Y_test)


print("✅ Saved: X_train.npy, X_test.npy, Y_train.npy, Y_test.npy, booleanized_data.npy")
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

✅ Saved: X_train.npy, X_test.npy, Y_train.npy, Y_test.npy, booleanized_data.npy
Train size: 20264, Test size: 5067


In [19]:
X_train[20]

array([0, 0, 0, ..., 0, 1, 0], shape=(19080,))

In [20]:
print(X_train.shape, Y_train.shape)
print(X_test.shape, Y_test.shape)

(20264, 19080) (20264,)
(5067, 19080) (5067,)


In [21]:
print("Train:", np.bincount(Y_train))
print("Test:", np.bincount(Y_test))

Train: [12748  7516]
Test: [3243 1824]


In [ ]:
# this below is for creating ham10000 as the same bins as above

In [26]:
# =========================
# HAM10000 TEST DATA
# =========================

ham_csv_files = glob.glob("/home/ubuntu/local/fractal_maps_ham_M32_S8/fractal_maps/*.csv")
ham_filenames = [os.path.splitext(os.path.basename(f))[0].replace('_fd', '') for f in ham_csv_files]

print("HAM10000 samples:", len(ham_filenames))
print(ham_filenames[:5])

HAM10000 samples: 10015
['ISIC_0026391', 'ISIC_0031121', 'ISIC_0028488', 'ISIC_0033714', 'ISIC_0032748']


In [28]:
booleanized_data_ham = {}

for f in ham_filenames:
    path = f"/home/ubuntu/local/fractal_maps_ham_M32_S8/fractal_maps/{f}_fd.csv"
    
    vals = np.loadtxt(path, delimiter=",").flatten()
    
    # 🔥 SAME discretizer from ISIC
    booleanized_data_ham[f] = booleanize_array(vals, discretizer).flatten()

print("HAM10000 booleanization done.")

HAM10000 booleanization done.


In [29]:
df_ham = pd.read_csv("/home/ubuntu/local/HAM10000/HAM10000_metadata.csv")

cancerous = ['akiec', 'bcc', 'mel']
non_canc = ['bkl', 'df', 'nv', 'vasc']

label_dict_ham = {}

for _, row in df_ham.iterrows():
    image_id = row['image_id']
    diagnosis = row['dx']
    
    if diagnosis in cancerous:
        label_dict_ham[image_id] = 1
    else:
        label_dict_ham[image_id] = 0

print("HAM labels ready.")

HAM labels ready.


In [30]:
X_test_ham = np.array([booleanized_data_ham[f] for f in ham_filenames], dtype=object)
Y_test_ham = np.array([label_dict_ham[f] for f in ham_filenames])

print("Shapes:", X_test_ham.shape, Y_test_ham.shape)

Shapes: (10015, 19080) (10015,)


In [31]:
np.save('X_test_ham.npy', X_test_ham, allow_pickle=True)
np.save('Y_test_ham.npy', Y_test_ham)

print("✅ Saved HAM test data")

✅ Saved HAM test data
